In [1]:
!pip install transformers seqeval evaluate accelerate -U

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [18]:
label_path = '/content/drive/MyDrive/datasetViMedNER/traindata/labels.txt'
train_path = '/content/drive/MyDrive/datasetViMedNER/traindata/train.txt'
dev_path = '/content/drive/MyDrive/datasetViMedNER/traindata/dev.txt'

with open(label_path, 'r', encoding='utf-8') as f:
    unique_tags = [line.strip() for line in f if line.strip()]

# Tạo lại từ điển ánh xạ
label2id = {tag: idx for idx, tag in enumerate(unique_tags)}
id2label = {idx: tag for idx, tag in enumerate(unique_tags)}

# convert to (word, tag)
def load_conll_data(file_path):
    sentences = []
    current_sentence = []

    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                if current_sentence:
                    sentences.append(current_sentence)
                    current_sentence = []
            else:
                parts = line.split()
                if len(parts) >= 2:
                    word, tag = parts[0], parts[1]
                    current_sentence.append((word, tag))
        if current_sentence:
            sentences.append(current_sentence)
    return sentences

# Đọc tập train, dev
train_sentences = load_conll_data(train_path)
dev_sentences = load_conll_data(dev_path)
final_training_sentences = train_sentences + dev_sentences

In [4]:
#chuẩn bị dataset cho model có thể xử lí
import torch
from torch.utils.data import Dataset
from transformers import AutoTokenizer

class ViMedNERDataset(Dataset):
    def __init__(self, sentences, tokenizer, label2id, max_length=128):
        self.sentences = sentences
        self.tokenizer = tokenizer
        self.label2id = label2id
        self.max_length = max_length

    def __len__(self):
        return len(self.sentences)

    def __getitem__(self, idx):
        sentence_data = self.sentences[idx]
        words = [item[0] for item in sentence_data]
        tags = [item[1] for item in sentence_data]

        # Tokenize từng từ và theo dõi số lượng sub-token của mỗi từ
        tokenized_outputs = []
        label_ids = []

        # Thêm token [CLS] đầu câu
        tokenized_outputs.append(self.tokenizer.cls_token_id)
        label_ids.append(-100)

        for word, tag in zip(words, tags):
            # Tokenize từng từ lẻ (không dùng is_split_into_words để tránh lỗi word_ids)
            sub_tokens = self.tokenizer.tokenize(word)
            sub_token_ids = self.tokenizer.convert_tokens_to_ids(sub_tokens)

            if len(sub_token_ids) > 0:
                # Sub-token đầu tiên nhận nhãn thật
                tokenized_outputs.append(sub_token_ids[0])
                label_ids.append(self.label2id[tag])

                # Các sub-token phía sau của cùng 1 từ nhận -100
                for sub_id in sub_token_ids[1:]:
                    tokenized_outputs.append(sub_id)
                    label_ids.append(-100)

        # Thêm token [SEP] cuối câu
        tokenized_outputs.append(self.tokenizer.sep_token_id)
        label_ids.append(-100)

        # Cắt ngắn (Truncation) nếu vượt quá max_length
        if len(tokenized_outputs) > self.max_length:
            tokenized_outputs = tokenized_outputs[:self.max_length]
            label_ids = label_ids[:self.max_length]

        # Tạo attention mask (1 cho token thật, 0 cho padding)
        attention_mask = [1] * len(tokenized_outputs)

        # Padding (Đệm) cho đủ max_length
        padding_length = self.max_length - len(tokenized_outputs)
        if padding_length > 0:
            tokenized_outputs = tokenized_outputs + [self.tokenizer.pad_token_id] * padding_length
            label_ids = label_ids + [-100] * padding_length
            attention_mask = attention_mask + [0] * padding_length

        item = {
            "input_ids": torch.tensor(tokenized_outputs, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
            "labels": torch.tensor(label_ids, dtype=torch.long)
        }
        return item

# 1. Khởi tạo Tokenizer
tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base-v2")

# 2. Khởi tạo lại Dataset
train_dataset = ViMedNERDataset(train_sentences, tokenizer, label2id)
dev_dataset = ViMedNERDataset(dev_sentences, tokenizer, label2id)

# train + dev dataset
final_training_dataset = train_dataset + dev_dataset


config.json:   0%|          | 0.00/678 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/895k [00:00<?, ?B/s]

bpe.codes:   0%|          | 0.00/1.14M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.13M [00:00<?, ?B/s]

In [5]:
import pickle
#lưu lại dataset
train_dataset_path = '/content/drive/MyDrive/datasetViMedNER/PreprocessedData/train_dataset.pkl'
dev_dataset_path = '/content/drive/MyDrive/datasetViMedNER/PreprocessedData/dev_dataset.pkl'

# Tiến hành lưu (serialize)
with open(train_dataset_path, 'wb') as f:
    pickle.dump(train_dataset, f)

with open(dev_dataset_path, 'wb') as f:
    pickle.dump(dev_dataset, f)

print("✅ Đã lưu thành công train_dataset và dev_dataset lên Google Drive!")

✅ Đã lưu thành công train_dataset và dev_dataset lên Google Drive!


In [14]:
import numpy as np
import pandas as pd
import evaluate
from sklearn.metrics import classification_report as sklearn_report
from IPython.display import display

seqeval = evaluate.load("seqeval")

def compute_eval_classify_metrics(pred):
    predictions, labels = pred
    predictions = np.argmax(predictions, axis=2)

    # 1. Lọc nhãn -100 và giữ nguyên cấu trúc List of Lists (dành cho Seqeval)
    true_predictions_seq = [
        [id2label[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels_seq = [
        [id2label[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    # 2. Trải phẳng thành mảng 1 chiều (dành cho Sklearn)
    true_predictions_flat = [tag for sent in true_predictions_seq for tag in sent]
    true_labels_flat = [tag for sent in true_labels_seq for tag in sent]

    metrics_dict = {}

    # =========================================================
    # BẢNG 1: ĐÁNH GIÁ THEO CỤM THỰC THỂ HOÀN CHỈNH (SEQEVAL)
    # =========================================================
    results_seq = seqeval.compute(predictions=true_predictions_seq, references=true_labels_seq)
    table_entity = []

    for key, value in results_seq.items():
        if isinstance(value, dict):
            table_entity.append({
                "Thực thể (Entity)": key,
                "Precision": f"{value['precision']:.4f}",
                "Recall": f"{value['recall']:.4f}",
                "F1-Score": f"{value['f1']:.4f}",
                "Number (Entities)": value['number']
            })
            metrics_dict[f"entity_{key}_f1"] = value["f1"]

    table_entity.append({
        "Thực thể (Entity)": "OVERALL",
        "Precision": f"{results_seq['overall_precision']:.4f}",
        "Recall": f"{results_seq['overall_recall']:.4f}",
        "F1-Score": f"{results_seq['overall_f1']:.4f}",
        "Number (Entities)": "-"
    })
    metrics_dict["overall_f1"] = results_seq['overall_f1']

    print("\n" + "="*75)
    print("📊 BẢNG 1: ĐÁNH GIÁ THEO CỤM THỰC THỂ (STRICT ENTITY LEVEL)")
    print("="*75)
    display(pd.DataFrame(table_entity))

    # =========================================================
    # BẢNG 2: ĐÁNH GIÁ CHI TIẾT TỪNG NHÃN RỜI RẠC (SKLEARN)
    # =========================================================
    report_tag = sklearn_report(true_labels_flat, true_predictions_flat, output_dict=True, zero_division=0)
    table_tag = []

    for key, value in report_tag.items():
        if key in ["accuracy", "macro avg", "weighted avg"]:
            continue
        table_tag.append({
            "Nhãn (Tag)": key,
            "Precision": f"{value['precision']:.4f}",
            "Recall": f"{value['recall']:.4f}",
            "F1-Score": f"{value['f1-score']:.4f}",
            "Number (Tokens)": int(value['support'])
        })

    overall_tag = report_tag["weighted avg"]
    table_tag.append({
        "Nhãn (Tag)": "OVERALL (Weighted)",
        "Precision": f"{overall_tag['precision']:.4f}",
        "Recall": f"{overall_tag['recall']:.4f}",
        "F1-Score": f"{overall_tag['f1-score']:.4f}",
        "Number (Tokens)": int(overall_tag['support'])
    })

    print("\n" + "="*75)
    print("📊 BẢNG 2: ĐÁNH GIÁ CHI TIẾT TỪNG NHÃN (TOKEN TAG LEVEL: B-, I-, O)")
    print("="*75)
    display(pd.DataFrame(table_tag))

    return metrics_dict

In [15]:
#trained model
import torch
from transformers import AutoModelForTokenClassification, AutoTokenizer, Trainer, TrainingArguments, DataCollatorForTokenClassification
import evaluate
seqeval = evaluate.load("seqeval") # Khai báo seqeval ở đây
from seqeval.metrics import classification_report

model_path =  "/content/drive/MyDrive/vimedner_final_baseline"
eval_tokenizer = AutoTokenizer.from_pretrained(model_path)
eval_model = AutoModelForTokenClassification.from_pretrained(model_path)

eval_args = TrainingArguments(
    output_dir="./eval_temp",
    per_device_eval_batch_size=32,   # Tăng batch size lên 32 để chạy nhanh gấp đôi
    report_to="none"
)

eval_trainer = Trainer(
    model=eval_model,
    args=eval_args,
    data_collator=DataCollatorForTokenClassification(tokenizer=eval_tokenizer),
    compute_metrics = compute_eval_classify_metrics
)

test_results = eval_trainer.predict(final_training_dataset)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


📊 BẢNG 1: ĐÁNH GIÁ THEO CỤM THỰC THỂ (STRICT ENTITY LEVEL)


,Thực thể (Entity),Precision,Recall,F1-Score,Number (Entities)
0,bien_phap_chan_doan,0.5988,0.7365,0.6606,1131
1,bien_phap_dieu_tri,0.5344,0.7198,0.6134,2334
2,nguyen_nhan_benh,0.1680,0.1576,0.1626,1104
3,ten_benh,0.7896,0.8784,0.8316,7161
4,trieu_chung_benh,0.5636,0.7044,0.6262,3038
5,OVERALL,0.6413,0.7528,0.6926,-



📊 BẢNG 2: ĐÁNH GIÁ CHI TIẾT TỪNG NHÃN (TOKEN TAG LEVEL: B-, I-, O)


,Nhãn (Tag),Precision,Recall,F1-Score,Number (Tokens)
0,B-bien_phap_chan_doan,0.7696,0.8152,0.7918,1131
1,B-bien_phap_dieu_tri,0.6937,0.8033,0.7445,2334
2,B-nguyen_nhan_benh,0.6735,0.0897,0.1583,1104
3,B-ten_benh,0.8503,0.9260,0.8866,7161
4,B-trieu_chung_benh,0.7166,0.7966,0.7545,3038
5,I-bien_phap_chan_doan,0.7629,0.8096,0.7856,3441
6,I-bien_phap_dieu_tri,0.7169,0.7299,0.7234,6421
7,I-nguyen_nhan_benh,0.5996,0.4830,0.5350,4331
8,I-ten_benh,0.8876,0.9485,0.9171,17992
9,I-trieu_chung_benh,0.7082,0.7725,0.7389,6258


In [19]:
import numpy as np

# 1. Chạy dự đoán trên tập dữ liệu muốn kiểm tra (ví dụ: test_dataset hoặc dev_dataset)
print("⏳ Đang thu thập dự đoán...")
predictions, labels, _ = test_results
predictions = np.argmax(predictions, axis=2)

# 2. Lọc bỏ các nhãn -100 và gom lại thành danh sách nhãn dự đoán cho từng câu
predicted_tags_per_sentence = []
for prediction, label in zip(predictions, labels):
    pred_sent = [id2label[p] for p, l in zip(prediction, label) if l != -100]
    predicted_tags_per_sentence.append(pred_sent)

# 3. Mở file để ghi kết quả đối chiếu (Ghi vào Drive để tải về xem cho tiện)
output_file = "/content/drive/MyDrive/datasetViMedNER/error_analysis_output.txt"

with open(output_file, "w", encoding="utf-8") as f:
    # Lặp qua từng câu gốc và câu dự đoán tương ứng (Lưu ý đổi test_sentences thành dev_sentences nếu đang chạy tập dev)
    for original_sent, predicted_tags in zip(final_training_sentences, predicted_tags_per_sentence):

        # Đảm bảo chiều dài 2 bên khớp nhau trước khi ghi
        if len(original_sent) == len(predicted_tags):
            for (word, true_tag), pred_tag in zip(original_sent, predicted_tags):
                # Ghi theo định dạng cột có tab cách quãng: Từ | Nhãn_Thật | Nhãn_Đoán
                f.write(f"{word}\t{true_tag}\t{pred_tag}\n")

            # Xuống dòng trống để ngăn cách các câu
            f.write("\n")
        else:
            print("⚠️ Cảnh báo: Lệch số lượng từ ở một câu, bỏ qua...")

print(f"✅ Đã xuất file thành công tại: {output_file}")

⏳ Đang thu thập dự đoán...
⚠️ Cảnh báo: Lệch số lượng từ ở một câu, bỏ qua...
⚠️ Cảnh báo: Lệch số lượng từ ở một câu, bỏ qua...
⚠️ Cảnh báo: Lệch số lượng từ ở một câu, bỏ qua...
⚠️ Cảnh báo: Lệch số lượng từ ở một câu, bỏ qua...
⚠️ Cảnh báo: Lệch số lượng từ ở một câu, bỏ qua...
⚠️ Cảnh báo: Lệch số lượng từ ở một câu, bỏ qua...
⚠️ Cảnh báo: Lệch số lượng từ ở một câu, bỏ qua...
⚠️ Cảnh báo: Lệch số lượng từ ở một câu, bỏ qua...
✅ Đã xuất file thành công tại: /content/drive/MyDrive/datasetViMedNER/error_analysis_output.txt


- model đang bị khó khăn trong việc đoán chuỗi thực thể do chỉ gắn linear+softmax trên đầu pretrained
- nhưng khả năng đoán nhãn rời rạc lại cao hơn vì chính lớp linear+softmax đó chỉ có khả năng đoán nhãn rời rạc không có cấu trúc
- các thực thể dài và ít xuất hiện hơn như nguyên nhân bệnh đều có tỉ lệ đoán rất thấp recall B-nguyen_nhan_benh là 0,0897 I-nguyen_nhan_benh cũng thấp
nguyên nhân bệnh là thực thể có thể phi cấu trúc hơn so với các thực thể khác(rất giống O)
- ngoài tên bệnh ra thì các chuỗi thực thể khác được đoán nhiều (recall cao) nhưng lại đoán sai (precision lại thấp) do thiếu các layer học cấu trúc của cụm entity
------------
* thêm crf: học quan hệ chuyển tiếp của chuỗi thực thể
* thay đổi focal loss để tránh học những từ xuất hiện nhiều và giảm đóng góp của nhãn O
* sau đó tinh chỉnh data để model học tốt hơn
